# Proyecto Integrador — Machine Learning  
## Módulo: Machine Learning | Soy Henry

**Estudiante:** Vanina Cavallin  
**Proyecto:** Predicción de Churn de Clientes  
**Empresa ficticia:** FinanceGuard  
**Avance:** Nº 4. Extracredit 

**Fecha:** 09/02/2026  

---

## 1) Importación de librerías y carga del dataset.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

DATA_PATH = "Churn_Modelling.csv"

df = pd.read_csv(DATA_PATH)
TARGET = 'Exited'
DROP_COLS = ['RowNumber', 'CustomerId', 'Surname']

X = df.drop(columns=[TARGET] + DROP_COLS)
y = df[TARGET].astype(int)

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

preprocess = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                               ('scaler', StandardScaler())]), num_cols),
        ('cat', Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                               ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_cols),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

best_model = Pipeline(steps=[
    ('preprocess', preprocess),
    ('model', GradientBoostingClassifier(random_state=42))
])

best_model.fit(X_train, y_train)
proba = best_model.predict_proba(X_test)[:, 1]
print('ROC-AUC:', roc_auc_score(y_test, proba))

C:\Users\vanin\AppData\Local\Temp\ipykernel_13480\1142144088.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=['object']).columns.tolist()


ROC-AUC: 0.8702176753024209


---


## 2) Optimización de threshold (métricas clásicas)
Evaluamos thresholds en una grilla y elegimos el mejor para la métrica objetivo.

In [2]:
def sweep_thresholds(y_true, proba, thresholds=None):
    if thresholds is None:
        thresholds = np.linspace(0.05, 0.95, 19)
    rows = []
    for t in thresholds:
        pred = (proba >= t).astype(int)
        rows.append({
            'threshold': t,
            'precision': precision_score(y_true, pred, zero_division=0),
            'recall': recall_score(y_true, pred, zero_division=0),
            'f1': f1_score(y_true, pred, zero_division=0),
        })
    return pd.DataFrame(rows)

df_thr = sweep_thresholds(y_test, proba)
df_thr.sort_values('f1', ascending=False).head(10)

,threshold,precision,recall,f1
5,0.30,0.627685,0.646192,0.636804
6,0.35,0.662162,0.601966,0.630631
4,0.25,0.569138,0.697789,0.626932
7,0.40,0.713376,0.550369,0.621359
3,0.20,0.511475,0.766585,0.613569
8,0.45,0.741259,0.520885,0.611833
9,0.50,0.782101,0.493857,0.605422
2,0.15,0.434728,0.842752,0.573579
10,0.55,0.815668,0.434889,0.567308
11,0.60,0.870968,0.398034,0.546374


In [3]:
best_f1_row = df_thr.loc[df_thr['f1'].idxmax()]
best_f1_thr = float(best_f1_row['threshold'])
print('Best F1 threshold:', best_f1_thr)
print(best_f1_row)

pred_best_f1 = (proba >= best_f1_thr).astype(int)
confusion_matrix(y_test, pred_best_f1)

Best F1 threshold: 0.3
threshold    0.300000
precision    0.627685
recall       0.646192
f1           0.636804
Name: 5, dtype: float64


array([[1437,  156],
       [ 144,  263]])

## 3) Optimización del punto de corte según costo de negocio
Ejemplo típico en churn:
- <b>FN</b> (no detectar un churn real) suele ser más caro que un <b>FP</b> (ofrecer un incentivo a alguien que no se iba).

Definimos:
- costo_FN = costo de perder un cliente
- costo_FP = costo de contactarlo / incentivo

Y buscamos el threshold que minimiza el costo esperado.

In [4]:
def cost_for_threshold(y_true, proba, thr, cost_fp=1.0, cost_fn=5.0):
    pred = (proba >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    return cost_fp*fp + cost_fn*fn, {'tn':tn,'fp':fp,'fn':fn,'tp':tp}

thresholds = np.linspace(0.05, 0.95, 37)
rows = []
COST_FP = 1.0
COST_FN = 5.0  # ajustá según tu supuesto de negocio

for t in thresholds:
    c, cm = cost_for_threshold(y_test, proba, t, cost_fp=COST_FP, cost_fn=COST_FN)
    rows.append({'threshold': t, 'cost': c, **cm})

cost_df = pd.DataFrame(rows)
best_cost_row = cost_df.loc[cost_df['cost'].idxmin()]
best_cost_row

threshold       0.15
cost          766.00
tn           1147.00
fp            446.00
fn             64.00
tp            343.00
Name: 4, dtype: float64

In [5]:
best_cost_thr = float(best_cost_row['threshold'])
print('Best cost threshold:', best_cost_thr)

pred_cost = (proba >= best_cost_thr).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, pred_cost).ravel()
print('CM:', {'tn':tn,'fp':fp,'fn':fn,'tp':tp})
print('Cost:', COST_FP*fp + COST_FN*fn)

# Matriz de confusión con costos (simple)
import pandas as pd

cm_cost = pd.DataFrame(
    [[tn, fp], [fn, tp]],
    index=['Actual 0 (No churn)','Actual 1 (Churn)'],
    columns=['Pred 0','Pred 1']
)
cm_cost

Best cost threshold: 0.15
CM: {'tn': np.int64(1147), 'fp': np.int64(446), 'fn': np.int64(64), 'tp': np.int64(343)}
Cost: 766.0


,Pred 0,Pred 1
Actual 0 (No churn),1147,446
Actual 1 (Churn),64,343


> ## Recomendación:
> - Si tu objetivo es <b>retener la mayor cantidad posible</b>, priorizá recall (y aceptá más FP).
> - Si tu presupuesto de campañas es limitado, optimizá costo con un ratio FN/FP razonable.
> - Reportá siempre el threshold elegido y la lógica de negocio detrás.
>